### Dataset sobre a Vacinação da Covid-19 na Microrregião de Anápolis, entre 2021 e 2023

Carregando o dataset:

In [13]:
import pandas as pd

df = pd.read_csv("../data/raw/vacinacovidgoias.csv", encoding="utf-8")

In [2]:
df.head()

,Município de vacinação,Idade Evento,Data da Notificação,Raça/Cor,Sexo,Imunobiológico (vacina),Dose,Gestante,Mês de gestação,Mulher amamentando no momento da vacinação?,Tipo de Evento,Reação / evento adverso,Classificação de gravidade,Gravidade,Desfecho (evolução do caso),Doenças (CID10) - Preexistente,Medicamento em uso anterior ou durante a vacinação,Nome do Medicamento,Houve atendimento médico?,Tipo de Atendimento
0,Abadiânia,18.0,2/16/2022,Branca,Masculino,1: Astrazeneca/Fiocruz,1: 1ª Dose,Não,0.0,Não,1: Erro de Imunização,1: Dose inadequada,1: Não grave,0,1: Cura sem sequelas,0,Não,Não,Não,0
1,Abadiânia,18.0,2/16/2022,Parda,Masculino,1: Astrazeneca/Fiocruz,1: 1ª Dose / 2: 2ª Dose / 3: 3ª dose,Não,0.0,Não,1: Erro de Imunização,1: Dose inadequada,1: Não grave,0,1: Cura sem sequelas,0,Não,Não,Não,0
2,Abadiânia,18.0,2/16/2022,Parda,Feminino,1: Astrazeneca/Fiocruz,1: 1ª Dose / 2: - 2ª Dose,Não,0.0,Não,1: Erro de Imunização,1: Dose inadequada,1: Não grave,0,1: Cura sem sequelas,0,Não,Não,Não,0
3,Abadiânia,18.0,2/16/2022,Branca,Feminino,1: Astrazeneca/Fiocruz,1: 1ª Dose / 2: 2ª Dose,Não,0.0,Não,1: Erro de Imunização,1: Dose inadequada,1: Não grave,0,1: Cura sem sequelas,0,Não,Não,Não,0
4,Abadiânia,18.0,2/16/2022,Branca,Feminino,1: Astrazeneca/Fiocruz,1: 1ª Dose,Não,0.0,Não,1: Erro de Imunização,1: Dose inadequada,1: Não grave,0,1: Cura sem sequelas,0,Não,Não,Não,0


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 452 entries, 0 to 451
Data columns (total 20 columns):
 #   Column                                              Non-Null Count  Dtype  
---  ------                                              --------------  -----  
 0   Município de vacinação                              444 non-null    object 
 1   Idade Evento                                        444 non-null    float64
 2   Data da Notificação                                 444 non-null    object 
 3   Raça/Cor                                            444 non-null    object 
 4   Sexo                                                444 non-null    object 
 5   Imunobiológico (vacina)                             444 non-null    object 
 6   Dose                                                443 non-null    object 
 7   Gestante                                            444 non-null    object 
 8   Mês de gestação                                     444 non-null    float64
 9  

#### Limpando o dataset

Padronizando nomes das colunas:
Durante a padronização, os caracteres especiais e acentos estavam sumindo das palavras.
Então, utilizei a biblioteca unicodedata para transformar os caracteres acentuados

In [ ]:
import re
import unicodedata

def limpar_colunas(col):
    # Converte para minúsculo e remove espaços nas extremidades
    col = col.lower().strip()
    
    # Normaliza para decompor caracteres acentuados (ex: 'á' vira 'a' + '´')
    # O formato NFKD separa o caractere base do acento
    col = unicodedata.normalize('NFKD', col)
    
    # Codifica em ASCII e ignora o que não for compatível (remove os acentos separados)
    col = col.encode('ascii', 'ignore').decode('utf-8')
    
    # Substitui espaços internos por underscores
    col = col.replace(" ", "_")
    
    # Remove qualquer caractere que não seja letra, número ou underscore
    col = re.sub(r'[^a-z0-9_]', '', col)
    
    return col

# Aplicando ao seu DataFrame
df.columns = [limpar_colunas(c) for c in df.columns]

In [17]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 452 entries, 0 to 451
Data columns (total 20 columns):
 #   Column                                              Non-Null Count  Dtype  
---  ------                                              --------------  -----  
 0   municipio_de_vacinacao                              444 non-null    object 
 1   idade_evento                                        444 non-null    float64
 2   data_da_notificacao                                 444 non-null    object 
 3   racacor                                             444 non-null    object 
 4   sexo                                                444 non-null    object 
 5   imunobiologico_vacina                               444 non-null    object 
 6   dose                                                443 non-null    object 
 7   gestante                                            444 non-null    object 
 8   mes_de_gestacao                                     444 non-null    float64
 9  

Convertendo datas, facilitando a busca por filtração por data

In [18]:
df['data_da_notificacao'] = pd.to_datetime(df['data_da_notificacao'], errors='coerce')

Convertendo idade, facilitando a agrupação e utilização de cálculos, como média

In [19]:
df['idade_evento'] = pd.to_numeric(df['idade_evento'], errors='coerce')

Algumas colunas estão utilizando "id" de maneira irrelevante para a análise, então iremos remover os ids:

In [21]:
# imunobiologico_vacina
df['imunobiologico_vacina'] = df['imunobiologico_vacina']\
    .str.split(":").str[1].str.strip()

In [22]:
# dose
df['dose'] = df['dose'].str.split("/").str[0]\
    .str.split(":").str[1].str.strip()

In [ ]:
# tipo_de_evento
df['tipo_de_evento'] = df['tipo_de_evento']\
    .str.split(":").str[1].str.strip()

In [23]:
# classificacao_de_gravidade
df['classificacao_de_gravidade'] = df['classificacao_de_gravidade']\
    .str.split(":").str[1].str.strip()

In [ ]:
#Verificando se tem valores nulos:
df.isnull().sum()

municipio_de_vacinacao                                  8
idade_evento                                            8
data_da_notificacao                                   226
racacor                                                 8
sexo                                                    8
imunobiologico_vacina                                   8
dose                                                    9
gestante                                                8
mes_de_gestacao                                         8
mulher_amamentando_no_momento_da_vacinacao              8
tipo_de_evento                                          8
reacao__evento_adverso                                 14
classificacao_de_gravidade                              9
gravidade                                               8
desfecho_evolucao_do_caso                               9
doencas_cid10__preexistente                            45
medicamento_em_uso_anterior_ou_durante_a_vacinacao    169
nome_do_medica

In [ ]:
# Removendo duplicados
df = df.drop_duplicates()

In [ ]:
# Resetando o indice, reorganizando os numeros das linhas
df = df.reset_index(drop=True)

Salvando o dataset limpo:

In [33]:
df.to_csv("../data/processed/vacinacovidgoias_limpo.csv", index=False)

In [ ]:
import pandas as pd
import re
import unicodedata

# 1. Função segura para padronizar os nomes das colunas
def padronizar_colunas(coluna):
    nfkd_form = unicodedata.normalize('NFKD', coluna)
    coluna_sem_acento = "".join([c for c in nfkd_form if not unicodedata.combining(c)])
    coluna_limpa = coluna_sem_acento.lower().strip().replace(" ", "_").replace("/", "_")
    return re.sub(r'[^a-z0-9_]', '', coluna_limpa)

# 2. Carregar o dataset
df = pd.read_csv("../data/raw/vacinacovidgoias.csv", encoding="utf-8")

# 3. Aplicar padronização nos nomes das colunas
df.columns = [padronizar_colunas(c) for c in df.columns]

# 4. Limpeza dos "IDs" (1:, 2:) SEM APAGAR o resto da informação
# Lista de colunas que costumam ter esse padrão numérico
colunas_com_id = [
    'imunobiologico_vacina', 
    'dose', 
    'tipo_de_evento', 
    'reacao__evento_adverso',
    'classificacao_de_gravidade',
    'desfecho_evolucao_do_caso',
    'doencas_cid10__preexistente'
]

for col in colunas_com_id:
    if col in df.columns:
        # A magia acontece aqui: o regex '\d+:\s*' encontra "um ou mais números, seguidos de ':' e espaços" e troca por nada ('').
        # Exemplo: "1: 1ª Dose / 2: 2ª Dose" vai virar "1ª Dose / 2ª Dose"
        df[col] = df[col].astype(str).apply(
            lambda x: re.sub(r'\d+:\s*', '', x) if x != 'nan' and pd.notnull(x) else x
        )

# 5. Converter a Idade
df['idade_evento'] = pd.to_numeric(df['idade_evento'], errors='coerce')

# 6. Tratar Datas de Forma Segura
# Datas apenas com ano (ex: "2021") virarão automaticamente "2021-01-01". 
# O errors='coerce' agora só vai anular o que for realmente um lixo irrecuperável.
df['data_da_notificacao'] = pd.to_datetime(df['data_da_notificacao'], errors='coerce')

# 7. Remover duplicados de forma segura
# Como as informações detalhadas foram preservadas, ele só vai apagar linhas que sejam 100% idênticas
df = df.drop_duplicates()

# 8. Resetar o index
df = df.reset_index(drop=True)

# Salvando o arquivo final limpo
df.to_csv('vacinacovidgoias_limpo_v2.csv', index=False, encoding='utf-8-sig')

# Visualizar o resultado para conferir
display(df.head())
df.info()

,municipio_de_vacinacao,idade_evento,data_da_notificacao,raca_cor,sexo,imunobiologico_vacina,dose,gestante,mes_de_gestacao,mulher_amamentando_no_momento_da_vacinacao,tipo_de_evento,reacao___evento_adverso,classificacao_de_gravidade,gravidade,desfecho_evolucao_do_caso,doencas_cid10__preexistente,medicamento_em_uso_anterior_ou_durante_a_vacinacao,nome_do_medicamento,houve_atendimento_medico,tipo_de_atendimento
0,Abadiânia,18.0,2022-02-16,Branca,Masculino,Astrazeneca/Fiocruz,1ª Dose,Não,0.0,Não,Erro de Imunização,1: Dose inadequada,Não grave,0,Cura sem sequelas,0,Não,Não,Não,0
1,Abadiânia,18.0,2022-02-16,Parda,Masculino,Astrazeneca/Fiocruz,1ª Dose / 2ª Dose / 3ª dose,Não,0.0,Não,Erro de Imunização,1: Dose inadequada,Não grave,0,Cura sem sequelas,0,Não,Não,Não,0
2,Abadiânia,18.0,2022-02-16,Parda,Feminino,Astrazeneca/Fiocruz,1ª Dose / - 2ª Dose,Não,0.0,Não,Erro de Imunização,1: Dose inadequada,Não grave,0,Cura sem sequelas,0,Não,Não,Não,0
3,Abadiânia,18.0,2022-02-16,Branca,Feminino,Astrazeneca/Fiocruz,1ª Dose / 2ª Dose,Não,0.0,Não,Erro de Imunização,1: Dose inadequada,Não grave,0,Cura sem sequelas,0,Não,Não,Não,0
4,Abadiânia,18.0,2022-02-16,Branca,Feminino,Astrazeneca/Fiocruz,1ª Dose,Não,0.0,Não,Erro de Imunização,1: Dose inadequada,Não grave,0,Cura sem sequelas,0,Não,Não,Não,0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 438 entries, 0 to 437
Data columns (total 20 columns):
 #   Column                                              Non-Null Count  Dtype         
---  ------                                              --------------  -----         
 0   municipio_de_vacinacao                              437 non-null    object        
 1   idade_evento                                        437 non-null    float64       
 2   data_da_notificacao                                 222 non-null    datetime64[ns]
 3   raca_cor                                            437 non-null    object        
 4   sexo                                                437 non-null    object        
 5   imunobiologico_vacina                               438 non-null    object        
 6   dose                                                438 non-null    object        
 7   gestante                                            437 non-null    object        
 8   mes_de_ges